# WiFi CSI Signal Exploration Notebook

> **Important note:** This notebook uses synthetic CSI-like data only. It is intended to demonstrate a research-prototype workflow for WiFi CSI fall detection. The results should not be interpreted as real-world, clinical, hardware-validated, or medical-grade fall-detection performance.

## Notebook goals

This notebook demonstrates a simple end-to-end prototype pipeline:

1. Generate synthetic CSI-like signals
2. Visualize amplitude and phase behavior
3. Simulate normal activity and fall-like events
4. Preprocess signals
5. Extract simple features
6. Train a baseline classifier
7. Evaluate prototype results
8. Add clinical-safety-aware metrics
9. Save figures and summaries

This notebook supports the broader research direction of secure WiFi CSI sensing for healthcare-oriented monitoring.

## 1. Import libraries

The notebook uses common scientific Python libraries: NumPy, pandas, matplotlib, SciPy, and scikit-learn.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.ndimage import uniform_filter1d
from scipy.fft import rfft, rfftfreq

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

os.makedirs('../figures', exist_ok=True)
os.makedirs('../results', exist_ok=True)

print('Libraries imported successfully.')

## 2. Generate synthetic CSI-like data

In a real WiFi sensing system, Channel State Information describes how WiFi signals are affected by the surrounding environment. Human movement can change CSI amplitude and phase patterns.

Here, we generate simplified synthetic CSI-like signals. A normal activity sample has smoother variations, while a fall-like event includes a short sudden disturbance.

In [ ]:
def generate_normal_activity_csi(num_packets=300, num_subcarriers=30, noise_level=0.05):
    """Generate synthetic CSI-like amplitude and phase for normal activity."""
    t = np.linspace(0, 1, num_packets)
    amplitude = np.zeros((num_packets, num_subcarriers))
    phase = np.zeros((num_packets, num_subcarriers))

    for sc in range(num_subcarriers):
        base_amp = 1.0 + 0.1 * np.sin(2 * np.pi * (2 + sc * 0.02) * t)
        slow_motion = 0.05 * np.sin(2 * np.pi * (5 + sc * 0.01) * t)
        amplitude[:, sc] = base_amp + slow_motion + noise_level * np.random.randn(num_packets)

        base_phase = 0.2 * np.sin(2 * np.pi * (3 + sc * 0.02) * t)
        phase[:, sc] = base_phase + noise_level * np.random.randn(num_packets)

    return amplitude, phase


def generate_fall_event_csi(num_packets=300, num_subcarriers=30, noise_level=0.05):
    """Generate synthetic CSI-like amplitude and phase with a fall-like disturbance."""
    amplitude, phase = generate_normal_activity_csi(num_packets, num_subcarriers, noise_level)

    event_center = int(num_packets * 0.55)
    event_width = int(num_packets * 0.06)
    event_window = np.arange(num_packets)
    disturbance = np.exp(-0.5 * ((event_window - event_center) / event_width) ** 2)

    for sc in range(num_subcarriers):
        scale = 0.4 + 0.2 * np.sin(sc)
        amplitude[:, sc] += scale * disturbance
        phase[:, sc] += 0.5 * scale * disturbance * np.sin(2 * np.pi * 10 * event_window / num_packets)

    return amplitude, phase


normal_amp, normal_phase = generate_normal_activity_csi()
fall_amp, fall_phase = generate_fall_event_csi()

print('Normal amplitude shape:', normal_amp.shape)
print('Fall-like amplitude shape:', fall_amp.shape)

## 3. Visualize CSI amplitude

The plot below compares the average CSI amplitude across subcarriers for normal activity and a synthetic fall-like event.

In [ ]:
time_axis = np.arange(normal_amp.shape[0])

plt.figure(figsize=(10, 5))
plt.plot(time_axis, normal_amp.mean(axis=1), label='Normal activity')
plt.plot(time_axis, fall_amp.mean(axis=1), label='Fall-like event')
plt.xlabel('Packet index')
plt.ylabel('Mean CSI amplitude')
plt.title('Synthetic CSI Amplitude: Normal vs Fall-like Event')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('../figures/synthetic_csi_amplitude.png', dpi=200)
plt.show()

## 4. Visualize CSI phase

The plot below compares the average CSI phase across subcarriers for normal activity and a synthetic fall-like event.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(time_axis, normal_phase.mean(axis=1), label='Normal activity')
plt.plot(time_axis, fall_phase.mean(axis=1), label='Fall-like event')
plt.xlabel('Packet index')
plt.ylabel('Mean CSI phase')
plt.title('Synthetic CSI Phase: Normal vs Fall-like Event')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('../figures/synthetic_csi_phase.png', dpi=200)
plt.show()

## 5. Compare one subcarrier

A single subcarrier view helps show how a sudden fall-like disturbance can appear as a localized change in the CSI time series.

In [ ]:
subcarrier_index = 10

plt.figure(figsize=(10, 5))
plt.plot(time_axis, normal_amp[:, subcarrier_index], label='Normal activity')
plt.plot(time_axis, fall_amp[:, subcarrier_index], label='Fall-like event')
plt.xlabel('Packet index')
plt.ylabel('CSI amplitude')
plt.title(f'Example Subcarrier {subcarrier_index}: Normal vs Fall-like Event')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('../figures/fall_vs_nonfall_example.png', dpi=200)
plt.show()

## 6. Preprocess synthetic CSI signals

This simple preprocessing step removes the mean, normalizes the signal, and applies a moving-average smoothing filter.

In [ ]:
def remove_mean(signal):
    """Remove the mean from each subcarrier."""
    return signal - np.mean(signal, axis=0, keepdims=True)


def normalize_csi(signal):
    """Normalize each subcarrier to zero mean and unit standard deviation."""
    mean = np.mean(signal, axis=0, keepdims=True)
    std = np.std(signal, axis=0, keepdims=True) + 1e-8
    return (signal - mean) / std


def smooth_signal(signal, window_size=5):
    """Apply moving-average smoothing along the packet/time axis."""
    return uniform_filter1d(signal, size=window_size, axis=0)


normal_amp_processed = smooth_signal(normalize_csi(remove_mean(normal_amp)))
fall_amp_processed = smooth_signal(normalize_csi(remove_mean(fall_amp)))

print('Preprocessing complete.')

## 7. Feature extraction

The next step extracts simple time-domain and frequency-domain features from each synthetic CSI sample.

In [ ]:
def extract_features(amplitude, sampling_rate=100):
    """Extract simple features from a CSI amplitude matrix."""
    mean_over_subcarriers = amplitude.mean(axis=1)

    features = {}
    features['mean'] = np.mean(mean_over_subcarriers)
    features['std'] = np.std(mean_over_subcarriers)
    features['max'] = np.max(mean_over_subcarriers)
    features['min'] = np.min(mean_over_subcarriers)
    features['range'] = features['max'] - features['min']
    features['energy'] = np.sum(mean_over_subcarriers ** 2)
    features['peak_to_average'] = features['max'] / (np.mean(np.abs(mean_over_subcarriers)) + 1e-8)

    spectrum = np.abs(rfft(mean_over_subcarriers))
    freqs = rfftfreq(len(mean_over_subcarriers), d=1 / sampling_rate)
    dominant_index = np.argmax(spectrum[1:]) + 1
    features['dominant_frequency'] = freqs[dominant_index]
    features['spectral_energy'] = np.sum(spectrum ** 2)

    return features


normal_features = extract_features(normal_amp_processed)
fall_features = extract_features(fall_amp_processed)

pd.DataFrame([normal_features, fall_features], index=['normal', 'fall_like'])

## 8. Create a synthetic dataset

This section creates multiple synthetic samples for two classes:

- `0`: normal activity
- `1`: fall-like event

These are synthetic labels for prototype testing only.

In [ ]:
def generate_synthetic_dataset(num_samples_per_class=150):
    """Generate a synthetic feature dataset for normal and fall-like classes."""
    rows = []
    labels = []

    for _ in range(num_samples_per_class):
        amp, _ = generate_normal_activity_csi(noise_level=np.random.uniform(0.03, 0.08))
        amp_processed = smooth_signal(normalize_csi(remove_mean(amp)))
        rows.append(extract_features(amp_processed))
        labels.append(0)

    for _ in range(num_samples_per_class):
        amp, _ = generate_fall_event_csi(noise_level=np.random.uniform(0.03, 0.08))
        amp_processed = smooth_signal(normalize_csi(remove_mean(amp)))
        rows.append(extract_features(amp_processed))
        labels.append(1)

    X = pd.DataFrame(rows)
    y = np.array(labels)
    return X, y


X, y = generate_synthetic_dataset(num_samples_per_class=150)

print('Feature matrix shape:', X.shape)
print('Labels shape:', y.shape)
X.head()

## 9. Train a baseline classifier

A simple Random Forest classifier is used as a baseline model. The purpose is to verify that the synthetic-data pipeline runs end to end, not to claim real-world fall-detection performance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=y
)

model = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print('Synthetic-data prototype metrics')
print('Accuracy:', round(accuracy, 3))
print('Precision:', round(precision, 3))
print('Recall:', round(recall, 3))
print('F1-score:', round(f1, 3))
print('\nClassification report:')
print(classification_report(y_test, y_pred, target_names=['normal', 'fall_like']))

## 10. Confusion matrix

The confusion matrix below summarizes the baseline classifier output on the synthetic test split.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title('Baseline Confusion Matrix\nSynthetic CSI-like Data Only')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.xticks([0, 1], ['normal', 'fall_like'])
plt.yticks([0, 1], ['normal', 'fall_like'])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center')

plt.colorbar()
plt.tight_layout()
plt.savefig('../figures/baseline_confusion_matrix.png', dpi=200)
plt.show()

## 11. Save baseline results

The following cell saves a short Markdown results summary into `results/baseline_results.md`.

In [ ]:
results_text = f'''# Baseline Results

## Important Note

These results are based on synthetic CSI-like data and are intended only to verify the prototype pipeline. They should not be interpreted as real-world fall-detection performance, clinical validation, hardware validation, or medical-grade accuracy.

## Dataset

- Data type: Synthetic CSI-like amplitude signals
- Classes: normal activity and fall-like event
- Samples: {len(X)} total synthetic samples
- Features: {X.shape[1]} extracted features

## Model

- Baseline classifier: Random Forest
- Train/test split: 75% / 25%
- Random seed: {RANDOM_SEED}

## Metrics

| Metric | Value |
|---|---:|
| Accuracy | {accuracy:.3f} |
| Precision | {precision:.3f} |
| Recall | {recall:.3f} |
| F1-score | {f1:.3f} |

## Interpretation

The baseline classifier verifies that the synthetic-data prototype pipeline can generate data, extract features, train a simple model, and produce evaluation outputs.

These numbers are not evidence of real-world fall-detection performance because the data are synthetic and simplified.

## Limitations

- No real CSI dataset is used in this notebook.
- No hardware deployment is included.
- No clinical or patient data are used.
- No medical-grade performance is claimed.
- The synthetic fall-like event is simplified and does not represent the full complexity of real falls.

## Next Steps

- Add more realistic synthetic channel models.
- Compare multiple baseline classifiers.
- Add clinical-safety-aware metrics such as missed falls, false alarms, and detection delay.
- Extend the repository toward adversarial robustness experiments.
- Evaluate on real CSI datasets in future work.
'''

with open('../results/baseline_results.md', 'w', encoding='utf-8') as f:
    f.write(results_text)

print('Saved results to ../results/baseline_results.md')

## 12. Clinical-Safety-Aware Evaluation

Accuracy alone is not enough for healthcare-oriented fall detection. A technically accurate model can still be unsafe if it misses important events, produces too many false alarms, or detects events too late.

This section maps basic ML errors to safety-oriented prototype metrics:

- **Missed fall:** a true fall-like event predicted as normal
- **False alarm:** a normal activity sample predicted as fall-like
- **Missed fall rate:** missed falls divided by total true fall-like events
- **False alarm rate:** false alarms divided by total true normal events
- **Alarm fatigue indicator:** simple prototype indicator based on false alarm rate
- **Detection delay:** simulated delay for correctly detected fall-like events

> These metrics are computed from synthetic prototype outputs only and are not clinical validation.

In [ ]:
def compute_clinical_safety_metrics(y_true, y_pred):
    """Compute simple clinical-safety-aware metrics from binary labels.

    Label convention:
    - 0: normal activity
    - 1: fall-like event
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    total_falls = np.sum(y_true == 1)
    total_normal = np.sum(y_true == 0)

    missed_falls = np.sum((y_true == 1) & (y_pred == 0))
    false_alarms = np.sum((y_true == 0) & (y_pred == 1))
    detected_falls = np.sum((y_true == 1) & (y_pred == 1))

    missed_fall_rate = missed_falls / total_falls if total_falls > 0 else 0.0
    false_alarm_rate = false_alarms / total_normal if total_normal > 0 else 0.0

    if false_alarm_rate >= 0.30:
        alarm_fatigue_indicator = 'High prototype false-alarm burden'
    elif false_alarm_rate >= 0.10:
        alarm_fatigue_indicator = 'Moderate prototype false-alarm burden'
    else:
        alarm_fatigue_indicator = 'Low prototype false-alarm burden'

    return {
        'total_true_fall_like_events': int(total_falls),
        'total_true_normal_events': int(total_normal),
        'detected_falls': int(detected_falls),
        'missed_falls': int(missed_falls),
        'false_alarms': int(false_alarms),
        'missed_fall_rate': missed_fall_rate,
        'false_alarm_rate': false_alarm_rate,
        'alarm_fatigue_indicator': alarm_fatigue_indicator
    }


clinical_metrics = compute_clinical_safety_metrics(y_test, y_pred)
clinical_metrics_df = pd.DataFrame([clinical_metrics])
clinical_metrics_df

## 13. Simulated detection delay

In a real fall-detection deployment, it is not enough to detect a fall eventually; the alert must occur quickly enough to support intervention.

Because this notebook uses synthetic sample-level classification rather than real streaming data, detection delay is simulated only for demonstration.

In [ ]:
correctly_detected_falls = clinical_metrics['detected_falls']

if correctly_detected_falls > 0:
    simulated_detection_delays_seconds = np.random.uniform(0.5, 4.0, size=correctly_detected_falls)
else:
    simulated_detection_delays_seconds = np.array([])

delay_summary = {
    'num_detected_falls_with_delay_estimate': int(correctly_detected_falls),
    'mean_detection_delay_seconds': float(np.mean(simulated_detection_delays_seconds)) if correctly_detected_falls > 0 else None,
    'max_detection_delay_seconds': float(np.max(simulated_detection_delays_seconds)) if correctly_detected_falls > 0 else None,
    'min_detection_delay_seconds': float(np.min(simulated_detection_delays_seconds)) if correctly_detected_falls > 0 else None
}

pd.DataFrame([delay_summary])

## 14. Save clinical-safety summary

The following cell saves a short Markdown summary into `results/clinical_safety_summary.md`.

In [ ]:
mean_delay = delay_summary['mean_detection_delay_seconds']
max_delay = delay_summary['max_detection_delay_seconds']
min_delay = delay_summary['min_detection_delay_seconds']

mean_delay_text = f'{mean_delay:.2f}' if mean_delay is not None else 'N/A'
max_delay_text = f'{max_delay:.2f}' if max_delay is not None else 'N/A'
min_delay_text = f'{min_delay:.2f}' if min_delay is not None else 'N/A'

clinical_summary_text = f'''# Clinical-Safety-Aware Summary

## Important Note

These metrics are computed from synthetic CSI-like prototype outputs only. They are intended to demonstrate the evaluation framework and should not be interpreted as clinical validation, medical-grade fall-detection performance, or real-world deployment evidence.

## Metrics

| Metric | Value |
|---|---:|
| Total true fall-like events | {clinical_metrics['total_true_fall_like_events']} |
| Total true normal events | {clinical_metrics['total_true_normal_events']} |
| Detected falls | {clinical_metrics['detected_falls']} |
| Missed falls | {clinical_metrics['missed_falls']} |
| False alarms | {clinical_metrics['false_alarms']} |
| Missed fall rate | {clinical_metrics['missed_fall_rate']:.3f} |
| False alarm rate | {clinical_metrics['false_alarm_rate']:.3f} |

## Alarm Fatigue Indicator

{clinical_metrics['alarm_fatigue_indicator']}

## Simulated Detection Delay

| Delay Metric | Value |
|---|---:|
| Mean detection delay seconds | {mean_delay_text} |
| Minimum detection delay seconds | {min_delay_text} |
| Maximum detection delay seconds | {max_delay_text} |

## Interpretation

The clinical-safety-aware metrics show how baseline ML outputs can be translated into safety-relevant indicators such as missed falls, false alarms, alarm fatigue, and detection delay.

These values are useful for demonstrating the evaluation framework, but they are not evidence of real-world or clinical performance.

## Limitations

- The data are synthetic CSI-like signals.
- The fall-like events are simplified.
- Detection delay is simulated, not measured from real-time hardware.
- No patient data, real clinical deployment, or medical-grade validation is included.

## Next Steps

- Apply these metrics to more realistic synthetic scenarios.
- Evaluate how adversarial CSI perturbations affect missed falls and false alarms.
- Connect robustness testing to safety outcomes.
- Validate with real CSI datasets in future work.
'''

with open('../results/clinical_safety_summary.md', 'w', encoding='utf-8') as f:
    f.write(clinical_summary_text)

print('Saved clinical-safety summary to ../results/clinical_safety_summary.md')

## 15. Final limitations and next steps

This notebook is a prototype demonstration only.

### Current limitations

- The data are synthetic CSI-like signals, not real WiFi CSI measurements.
- No clinical data are used.
- No hardware deployment is included.
- The fall-like event model is simplified.
- The baseline classifier is used only to verify the workflow.
- Clinical-safety metrics are computed from synthetic labels and predictions only.

### Future work

- Add real CSI datasets when available.
- Improve the synthetic signal model.
- Add additional preprocessing methods.
- Expand clinical-safety-aware metrics.
- Add adversarial robustness experiments.
- Connect robustness results to missed falls, false alarms, and detection delay.
- Connect this prototype to secure WiFi sensing research.